# LifeMate Caller SFT: top 8, one epoch with resumable sessions

Train-only compact schemas, private train/validation split. Defaults: 4,096 tokens, one T4, batch 1, accumulation 8, 1 epoch, LR 5e-5, checkpoints every 10 steps. A session saves and stops after 8 hours; resume from its full checkpoint to finish the epoch. No Planner/RL training. Keep outputs private. Live query logging off.


In [ ]:
REPO_URL = "https://github.com/AmnO-O/ZolaAIC.git"
REVISION = "48fd0581008e43e4063c353bb4c6fc4af07aa39e"  # Pin the training source commit, including when resuming.
REPO_DIR = "/kaggle/working/ZolaAIC"
TRAIN_DIR = "/kaggle/input/datasets/amnpham/zalochallenge/train+public_test/train"
MODEL_PATH = ""
MAX_LENGTH = 4096
MAX_STEPS = -1
EPOCHS = 1
RETRIEVAL_TOP_K = 8
LEARNING_RATE = 5e-5
SAVE_STEPS = 10
EVAL_STEPS = 100
MAX_RUN_SECONDS = 8 * 60 * 60
RESUME_CHECKPOINT = ""  # Complete checkpoint directory from your previous private output.


In [ ]:
import json
import os
os.environ.update(PYTORCH_ALLOC_CONF="expandable_segments:True", CUDA_VISIBLE_DEVICES="0", PYTHONHASHSEED="42", CUBLAS_WORKSPACE_CONFIG=":4096:8")
import os
from pathlib import Path
import subprocess
import sys

repo = Path(REPO_DIR).resolve()
if repo.exists():
    actual_root = Path(subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"], cwd=repo, text=True).strip()).resolve()
    if actual_root != repo:
        raise RuntimeError("REPO_DIR must point to the repository root")
    remote = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=repo, text=True).strip()
    if remote != REPO_URL:
        raise RuntimeError(f"Unexpected repository at {repo}; choose a new REPO_DIR")
    dirty = subprocess.check_output(
        ["git", "status", "--porcelain"], cwd=repo, text=True).strip()
    if dirty:
        raise RuntimeError("Checkout has local edits. Preserve them and choose a new REPO_DIR.")
else:
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)

# Works for a remote branch, tag, or exact commit; never merges local changes.
subprocess.run(["git", "fetch", "origin", REVISION], cwd=repo, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=repo, check=True)
if not (repo / "src/lifemate/prompt_assets/system.txt").is_file():
    raise RuntimeError("This revision predates the clean pipeline; select its notebook or a newer revision.")

COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
print("Source commit:", COMMIT)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(repo / "requirements-train.txt")], check=True)


In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


In [ ]:
import os
import uuid
from pathlib import Path
if not torch.cuda.is_available():
    raise RuntimeError("Select a Kaggle GPU before training")
train_dir = Path(TRAIN_DIR)
if not (train_dir / "train.jsonl").is_file():
    candidates = [Path("/kaggle/input/datasets/amnpham/zalochallenge/train+public_test/train"), Path("/kaggle/input/zalochallenge/train+public_test/train")]
    train_dir = next((p for p in candidates if (p / "train.jsonl").is_file()), None)
    if train_dir is None:
        raise FileNotFoundError("Set TRAIN_DIR to the train split; public/private test is forbidden")
if not MODEL_PATH:
    subprocess.run([sys.executable, "scripts/download_models.py", "--component", "llm"], cwd=repo, check=True)
model_path = Path(MODEL_PATH).resolve() if MODEL_PATH else repo / "models" / "qwen3.5-4b"
run_dir = Path("/kaggle/working/training") / uuid.uuid4().hex[:12]
run_dir.mkdir(parents=True)
data_dir = run_dir / "data"
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", PYTHONHASHSEED="42", CUBLAS_WORKSPACE_CONFIG=":4096:8")
subprocess.run([sys.executable, "scripts/prepare_sft.py", "--train-dir", str(train_dir),
                "--output", str(data_dir), "--retrieval-top-k", str(RETRIEVAL_TOP_K)], cwd=repo, check=True)
manifest = json.loads((data_dir / "manifest.json").read_text())
print("Train / validation samples:", len(manifest["train"]), len(manifest["validation"]))
print("Rejected trajectories:", len(manifest["rejected"]))
print("Output:", run_dir)

print("Train-only target repairs:", manifest["retrieval"]["train_target_repairs"])
print("Validation targets omitted from loss:", len(manifest["retrieval"]["validation_missing_target"]))


In [ ]:
command = [sys.executable, "scripts/train_adapter.py", "--model", str(model_path),
           "--data", str(data_dir), "--output", str(run_dir / "training"),
           "--max-length", str(MAX_LENGTH), "--epochs", str(EPOCHS), "--max-steps", str(MAX_STEPS),
           "--learning-rate", str(LEARNING_RATE), "--save-steps", str(SAVE_STEPS),
           "--eval-steps", str(EVAL_STEPS), "--max-run-seconds", str(MAX_RUN_SECONDS),
           "--base-revision", json.loads((repo / "configs/baseline.json").read_text())["llm"]["revision"]]
if RESUME_CHECKPOINT:
    command += ["--resume-from-checkpoint", str(Path(RESUME_CHECKPOINT).resolve())]
if MAX_STEPS == 1:
    command.append("--smoke")
with (run_dir / "training.log").open("w", encoding="utf-8") as log:
    process = subprocess.Popen(command, cwd=repo, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, env={**os.environ, "PYTHONUNBUFFERED":"1"})
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
    returncode = process.wait()
if returncode:
    raise RuntimeError("Training failed; inspect training.log and skipped_length.json")
print("Adapter:", run_dir / "training" / "adapter")
print("Validation loss is not tool-use accuracy. A one-step smoke adapter is not a trained solution.")

report = json.loads((run_dir / "training" / "training_report.json").read_text())
print({k:report.get(k) for k in ["status", "global_step", "changed_adapter_tensors", "max_adapter_delta", "peak_allocated_bytes", "trainable_parameters", "retained_examples"]})

print("Epoch completed:", report.get("epoch_completed"))
print("Latest resumable checkpoint:", report.get("latest_checkpoint"))
print("Resume uses optimizer, scheduler, scaler and RNG state; loading adapter alone is not equivalent.")


## Interpretation

At 4,096 tokens, this top-8 preparation retains 3,938 train targets and 540 validation-loss targets. The 266 overlength train targets are explicitly omitted, as selected by the user; no prompt truncation. There are also 107 validation targets missing a schema and 17 overlength covered validation targets. Loss is therefore not full pipeline accuracy.

A checkpointed_time_budget status is a partial epoch. Attach the previous private notebook output, set RESUME_CHECKPOINT to its complete checkpoint-N directory and keep the source commit/data/settings identical. The resume identity refuses incompatible or incomplete state. The LR schedule targets the entire epoch across sessions.

After the epoch completes, compare frozen base and adapter using real retrieval on all validation samples. No gold schema repair or dropping failed samples in task scoring. Model loading and teacher-forced validation loss do not establish tool accuracy or the official runtime.
